<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Machine-Learning/20-ml-systems-research-practice.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Machine Learning guideline](Machine Learning.html)


## **Machine Learning Systems and Research Practice**

A fitted model is a mathematical object. A **machine learning system** is the complete mechanism that turns observations into actions and then survives contact with changing data, users, infrastructure, and organizational decisions. It includes data collection, label construction, feature computation, training, evaluation, artifact storage, serving, monitoring, fallback behavior, human review, and the feedback generated by earlier predictions. A model can be statistically strong while the surrounding system is unreliable.

This distinction changes the central question. Model development asks:

> Which function $f_\theta(x)$ predicts the target well on an appropriate evaluation distribution?

System development asks:

> Under what data, operational, and decision contract does the complete pipeline create net value without violating its constraints?

One useful abstraction is constrained expected utility:

$$
\max_{\pi,\,f_\theta}
\quad
\mathbb{E}\left[
B\bigl(Y,\pi(f_\theta(X))\bigr)
-C_{\text{decision}}
-C_{\text{operation}}
-C_{\text{harm}}
\right]
$$

subject to requirements such as

$$
\text{latency}_{p99}\le L_{\max},
\qquad
\text{availability}\ge A_{\min},
\qquad
\text{memory}\le M_{\max},
\qquad
g_k(\text{slice metrics})\le \tau_k.
$$

Here $f_\theta$ produces a score or prediction, while $\pi$ is the **decision policy** that converts it into an action, perhaps using a threshold, abstention region, capacity limit, or human review. The objective includes benefits and several kinds of cost. The constraints are not decorative reporting metrics: a candidate that violates a hard safety, latency, privacy, or resource requirement is infeasible even if its average test score is highest.

The following official diagram captures the first systems lesson: production model code is surrounded by data verification, configuration, resource management, serving, monitoring, and process tooling.

<div class="diagram-scroll">

![A production machine learning system contains many components beyond model code.](assets/google-production-ml-system-official.png){fig-alt="Google diagram showing ML model code as one small component among data collection, verification, feature extraction, configuration, monitoring, serving, and resource management."}

</div>

*Image source: [Google Machine Learning Crash Course, Production ML systems](https://developers.google.com/machine-learning/crash-course/production-ml-systems), licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).*

The model's offline metric can fail to represent system quality for several reasons:

- the training label is only a proxy for the real outcome;
- the serving path computes a different feature than the training path;
- a p99 latency spike causes timeouts for exactly the busiest user segment;
- the action changes which labels become observable;
- a newer model improves average accuracy but overloads memory or increases cost;
- labels arrive weeks later, so performance degradation remains invisible;
- an apparently successful experiment cannot be reproduced from its code, data, and configuration.

Consequently, the unit of correctness is the **end-to-end contract**, not the estimator in isolation.

<div class="diagram-scroll">

![The machine learning lifecycle closes the loop from framing through operation.](assets/ml-system-lifecycle.svg){fig-alt="Five stages show framing, data, experimentation, release, and operation, with production evidence feeding back to framing."}

</div>

The lifecycle produces several distinct artifacts:

| Stage | Main question | Durable artifact |
|---|---|---|
| Frame | What decision is being improved, for whom, under which constraints? | Decision contract and baseline |
| Data | What was knowable at prediction time, and how is the label generated? | Data contract, lineage, validation report |
| Experiment | Which controlled comparison supports the claimed improvement? | Run records, predictions, uncertainty, ablations |
| Release | Can the candidate be packaged, tested, rolled out, and reversed? | Versioned model package and deployment plan |
| Operate | Is the deployed decision process still healthy and valuable? | Monitoring, incidents, delayed-label evaluation, retirement record |

<details>
<summary><strong>Python: rank candidates by utility while enforcing hard system constraints</strong></summary>

```python
from dataclasses import dataclass


@dataclass(frozen=True)
class Candidate:
    name: str
    true_positives: int
    false_positives: int
    false_negatives: int
    p99_latency_ms: float
    availability: float
    cost_per_1000: float
    worst_slice_recall: float


def assess(candidate: Candidate) -> dict:
    # The utility values belong to the decision context, not to the classifier.
    benefit = 8.0 * candidate.true_positives
    decision_cost = 2.0 * candidate.false_positives + 5.0 * candidate.false_negatives
    operating_cost = candidate.cost_per_1000 * 40.0  # expected daily volume in thousands
    utility = benefit - decision_cost - operating_cost

    checks = {
        "p99_latency": candidate.p99_latency_ms <= 80.0,
        "availability": candidate.availability >= 0.999,
        "slice_recall": candidate.worst_slice_recall >= 0.72,
    }
    return {
        "name": candidate.name,
        "utility": round(utility, 1),
        "feasible": all(checks.values()),
        "failed_constraints": [name for name, passed in checks.items() if not passed],
    }


candidates = [
    Candidate("heuristic", 620, 150, 210, 4, 0.9999, 0.02, 0.74),
    Candidate("compact_model", 710, 170, 120, 24, 0.9997, 0.35, 0.78),
    Candidate("large_ensemble", 745, 155, 85, 132, 0.9987, 4.80, 0.81),
]

for result in map(assess, candidates):
    print(result)
```

</details>

The large ensemble has the strongest predictive counts, but it is not a valid release candidate under the stated latency and availability contract. This is not a reason to hide its result. It is evidence for the next engineering question: compress it, precompute its output, reduce its feature path, relax the decision deadline with stakeholder approval, or retain the compact model.

**Comparison.** A notebook model is optimized against a dataset. A machine learning system is optimized against a decision process and maintained under change. Model metrics remain essential, but they become one layer of evidence inside a larger reliability argument.


### **Problem Framing and System Boundaries**

Problem framing converts a vague request such as “predict churn” or “detect fraud” into a falsifiable and operational contract. The contract must name the **decision unit**, **prediction time**, **target horizon**, **action**, **affected population**, **label mechanism**, **baseline policy**, **capacity**, and **failure costs**. If any of these are ambiguous, two teams can build accurate models for different problems while believing they are solving the same one.

<div class="diagram-scroll">

![Problem framing links users and observations to decisions, outcomes, and evidence.](assets/problem-framing-contract.svg){fig-alt="A flow from user state to prediction, decision, outcome, and evidence emphasizes that a predictive metric must connect to an action."}

</div>

#### **Users, Decisions, Constraints, and Success Metrics**

A robust framing exercise answers the following questions before model selection:

| Contract element | Precise question | Common failure |
|---|---|---|
| Decision unit | Is one row a user, account, session, transaction, image, or time window? | Leakage across rows belonging to the same entity |
| Prediction time | At exactly what moment must the prediction be available? | Features computed after the decision point |
| Target and horizon | What event is predicted, and within what period? | Mixing short- and long-horizon labels |
| Action | What changes when the score is high or low? | Building a prediction that nobody can act on |
| Capacity | How many cases can the downstream process handle? | Optimizing a threshold that exceeds review capacity |
| Errors and harm | What is the cost of false positives, false negatives, abstention, and delay? | Treating all mistakes as interchangeable |
| Baseline | What currently happens without the model? | Claiming gain relative to no meaningful comparator |
| Feedback | Does the action change exposure, behavior, or label observability? | Training on selectively observed outcomes |
| Constraints | What latency, availability, memory, privacy, and slice requirements are hard limits? | Selecting an infeasible model after experimentation |

A **success metric** should be close enough to the real objective to guide improvement and stable enough to measure repeatedly. Often no single metric satisfies both goals, so a system uses:

- a **primary decision metric**, such as expected savings, cases correctly prioritized within capacity, or time to successful resolution;
- **model diagnostics**, such as log loss, AUROC, calibration, recall at a fixed workload, or ranking quality;
- **guardrails**, such as p99 latency, subgroup false-negative rate, complaint rate, memory, or cost;
- **leading indicators**, available quickly but imperfectly related to the final outcome;
- **lagging outcomes**, more trustworthy but delayed.

The distinction between an **objective**, a **metric**, and a **constraint** matters. The training objective is the differentiable quantity optimized by the algorithm. The evaluation metric estimates a property of predictions. The system objective represents decision value. A hard constraint defines feasibility. Improving one does not guarantee improvement in the others.

#### **When Not to Use Machine Learning**

Machine learning is not automatically appropriate when a problem contains data. Prefer a deterministic rule, query, optimization routine, or human process when:

- the desired behavior can be specified exactly and changes rarely;
- there are too few representative examples or no credible label mechanism;
- errors are intolerable and cannot be bounded, detected, reviewed, or reversed;
- the action cannot change, so a more accurate prediction creates no value;
- the environment changes faster than labels and retraining can follow;
- the true requirement is causal intervention, database retrieval, constraint satisfaction, or anomaly investigation rather than prediction;
- a simple heuristic already meets the decision and operational contract;
- the organization cannot own monitoring, incident response, access control, and retirement.

The correct comparison is not “machine learning versus doing nothing.” It is **machine learning versus the best maintainable non-ML alternative**. A fixed rule can be more transparent, testable, cheap, and stable. Conversely, a growing collection of interacting manual rules may become harder to maintain than a supervised model trained from feedback.

<details>
<summary><strong>Python: compare policies using decision value and review capacity</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(20)
n = 5000

# Simulated latent event probability and binary outcome.
risk = rng.beta(2.0, 8.0, size=n)
y = rng.binomial(1, risk)

# Three policies: no review, a cheap heuristic, and an ML score.
heuristic_score = np.clip(risk + rng.normal(0, 0.18, n), 0, 1)
model_score = np.clip(risk + rng.normal(0, 0.09, n), 0, 1)

capacity = 500
benefit_if_caught = 12.0
review_cost = 1.5


def top_capacity_value(name, score):
    selected = np.argsort(score)[-capacity:]
    caught = int(y[selected].sum())
    value = benefit_if_caught * caught - review_cost * capacity
    return {"policy": name, "reviewed": capacity, "events_caught": caught, "value": value}


print({"policy": "no_review", "reviewed": 0, "events_caught": 0, "value": 0.0})
print(top_capacity_value("heuristic", heuristic_score))
print(top_capacity_value("ml_model", model_score))
```

</details>

This example evaluates each policy at the same downstream capacity. A threshold such as $0.5$ would be arbitrary because the team can review only 500 cases. The appropriate system metric is therefore **events caught within capacity**, followed by net value after review cost.

### **Baseline-First Development**

A baseline is an implemented comparator that tests whether the project has learned anything useful. It also exposes label mistakes, split leakage, metric bugs, and integration costs before model complexity obscures them. A good baseline ladder increases complexity one justified step at a time.

<div class="diagram-scroll">

![A baseline ladder requires each increase in complexity to produce controlled evidence.](assets/baseline-evidence-ladder.svg){fig-alt="A rising sequence from dummy and heuristic baselines through classical and candidate models to a deployable system."}

</div>

#### **Dummy, Heuristic, and Classical Baselines**

- A **dummy baseline** predicts prevalence, the mean, the median, a random ranking, or a seasonal value. It checks whether the metric and split behave sensibly.
- A **heuristic baseline** encodes current domain knowledge or the existing policy. It is often the true operational comparator.
- A **classical baseline** uses a transparent, inexpensive model such as regularized linear or logistic regression, a shallow tree, nearest neighbors, or a simple time-series method.
- A **system baseline** includes the current data pipeline, serving path, latency, cost, and outcome. It prevents a model-only comparison from ignoring integration.

A baseline is not deliberately weak. If a stronger, well-tuned baseline is readily available, using a weak one exaggerates the contribution. Hyperparameter budget, preprocessing, data access, and evaluation protocol should be comparable across candidates.

#### **Ablations and Controlled Comparisons**

An **ablation** removes or replaces one component while holding the rest of the protocol fixed. If a system contains a new feature group, loss term, sampler, model block, and post-processing rule, the final score alone cannot reveal which component caused the gain. A controlled ablation changes one factor and reruns the complete evaluation.

For paired test examples, define the per-example difference between candidate $A$ and baseline $B$:

$$
d_i=\ell\bigl(y_i,\hat y_i^{(B)}\bigr)
-\ell\bigl(y_i,\hat y_i^{(A)}\bigr).
$$

Then $\bar d>0$ favors $A$. Because both models are evaluated on the same examples, uncertainty should preserve that pairing, for example with a paired bootstrap over examples or groups. Multiple seeds are still needed when training randomness is material.

<details>
<summary><strong>Python: run a baseline ladder and a controlled feature ablation</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = make_classification(
    n_samples=5000,
    n_features=12,
    n_informative=6,
    n_redundant=2,
    weights=[0.78, 0.22],
    class_sep=1.0,
    random_state=20,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.35, stratify=y, random_state=20
)

models = {
    "dummy": DummyClassifier(strategy="prior"),
    "logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "forest": RandomForestClassifier(
        n_estimators=180, min_samples_leaf=10, n_jobs=-1, random_state=20
    ),
}

for name, estimator in models.items():
    estimator.fit(X_train, y_train)
    score = estimator.predict_proba(X_test)[:, 1]
    print(name, round(roc_auc_score(y_test, score), 4))

# Controlled ablation: same estimator, rows, seed, and metric; only the final
# four feature columns are removed.
full = RandomForestClassifier(
    n_estimators=180, min_samples_leaf=10, n_jobs=-1, random_state=20
).fit(X_train, y_train)
ablated = RandomForestClassifier(
    n_estimators=180, min_samples_leaf=10, n_jobs=-1, random_state=20
).fit(X_train[:, :-4], y_train)

full_auc = roc_auc_score(y_test, full.predict_proba(X_test)[:, 1])
ablated_auc = roc_auc_score(y_test, ablated.predict_proba(X_test[:, :-4])[:, 1])
print("feature_group_contribution", round(full_auc - ablated_auc, 4))
```

</details>

**Comparison.** Problem framing specifies what success means; baselines determine whether learning improves on a credible alternative; ablations identify which added component earns the improvement. Together they prevent a high benchmark score from becoming an unsupported system claim.


### **Data and Feature Pipelines**

A data pipeline is the versioned process that converts source events into training examples and serving inputs. Its correctness is temporal and semantic, not merely syntactic. Two tables can share the same columns and types while representing different populations, time windows, units, or states of knowledge.

The pipeline should preserve at least four forms of provenance:

1. **Origin:** source system, owner, collection mechanism, and access policy.
2. **Semantics:** entity, unit, timestamp meaning, missing-value meaning, and valid range.
3. **Transformation:** code version, parameters, dependencies, and upstream inputs.
4. **Time:** event time, ingestion time, processing time, prediction time, and label maturity.

A **data contract** makes these assumptions executable. It can require a schema, uniqueness, null policy, category vocabulary, timestamp ordering, freshness, range, and cross-column invariants. Contract tests should run at ingestion, before training, and on the serving path. Statistical checks such as a distribution distance complement the contract but do not replace semantic validation: a column measured in dollars can silently become cents while remaining numeric and smooth.

#### **Batch and Streaming Data**

**Batch processing** operates on bounded collections, such as all transactions from the previous day. It is easier to replay, test, aggregate, and correct. Its cost is staleness: a feature may be hours old when the decision is made.

**Streaming processing** updates state as events arrive. It supports low-latency decisions and rapidly changing features, but introduces ordering, duplication, late events, checkpointing, and state-recovery problems. A stream therefore needs explicit event identifiers, watermarks, idempotent updates, and a late-data policy.

Three timestamps must not be conflated:

- **event time:** when the real-world event occurred;
- **ingestion time:** when the platform received it;
- **processing time:** when a computation handled it.

For event $e$, ingestion delay is

$$
\Delta_{\text{ingest}}(e)=t_{\text{ingest}}(e)-t_{\text{event}}(e).
$$

If the 99th percentile delay is two hours, a “transactions in the previous hour” feature cannot be treated as complete at prediction time without an explicit correction or watermark. Backfilled events can improve historical tables while making them unlike the partial state that existed online.

<details>
<summary><strong>Python: enforce a compact data contract before training</strong></summary>

```python
import pandas as pd

events = pd.DataFrame(
    {
        "event_id": ["e1", "e2", "e3", "e4"],
        "account_id": [101, 101, 102, 103],
        "event_time": pd.to_datetime(
            ["2026-04-01 09:00", "2026-04-01 10:00", "2026-04-01 09:30", "2026-04-01 10:10"],
            utc=True,
        ),
        "ingestion_time": pd.to_datetime(
            ["2026-04-01 09:03", "2026-04-01 10:07", "2026-04-01 09:31", "2026-04-01 10:12"],
            utc=True,
        ),
        "amount": [42.5, 18.0, 120.0, 7.5],
        "channel": ["web", "mobile", "web", "branch"],
    }
)


def validate_event_contract(frame: pd.DataFrame) -> list[str]:
    errors = []
    required = {
        "event_id", "account_id", "event_time",
        "ingestion_time", "amount", "channel",
    }
    missing = required - set(frame.columns)
    if missing:
        errors.append(f"missing columns: {sorted(missing)}")
        return errors

    if frame["event_id"].duplicated().any():
        errors.append("event_id must be unique")
    if frame[["account_id", "event_time", "amount"]].isna().any().any():
        errors.append("entity, event time, and amount cannot be null")
    if (frame["amount"] < 0).any() or (frame["amount"] > 1_000_000).any():
        errors.append("amount outside the documented range")
    if not set(frame["channel"]).issubset({"web", "mobile", "branch"}):
        errors.append("unknown channel")
    if (frame["ingestion_time"] < frame["event_time"]).any():
        errors.append("ingestion cannot precede event time")

    delay_minutes = (
        frame["ingestion_time"] - frame["event_time"]
    ).dt.total_seconds() / 60
    if delay_minutes.quantile(0.99) > 30:
        errors.append("p99 ingestion delay exceeds the 30-minute freshness contract")
    return errors


violations = validate_event_contract(events)
print("contract_status", "PASS" if not violations else "FAIL")
print("violations", violations)
```

</details>

#### **Point-in-Time Correctness and Label Construction**

A training row should contain only information that was knowable when its historical prediction would have been made. For entity $i$ with prediction time $t_i$, a feature event at time $s$ is eligible only if

$$
s\le t_i
$$

and, in a realistic replay, if it had also been ingested and processed by $t_i$. A **point-in-time join** retrieves the latest eligible value or aggregates a window ending at $t_i$. A conventional database join that uses the latest value in the table can leak future information into every historical row.

<div class="diagram-scroll">

![Point-in-time joins reconstruct the information available at a historical prediction time.](assets/point-in-time-data.svg){fig-alt="Feature events appear on a timeline around a prediction time; only events known before the boundary are allowed in the training row."}

</div>

Labels require the same discipline. For a target such as “default within 30 days,” the dataset must:

- define the start and end of the outcome window;
- exclude or mark examples whose 30-day window has not matured;
- distinguish a true negative from a censored or missing outcome;
- record whether an action prevented the event from being observed;
- avoid features derived from post-outcome investigation or resolution.

<details>
<summary><strong>Python: construct a point-in-time-correct feature with an as-of join</strong></summary>

```python
import pandas as pd

feature_events = pd.DataFrame(
    {
        "account_id": [1, 1, 1, 2, 2],
        "event_time": pd.to_datetime(
            ["2026-01-02", "2026-01-09", "2026-01-20", "2026-01-05", "2026-01-18"],
            utc=True,
        ),
        "balance": [100, 80, 20, 200, 160],
    }
)

prediction_rows = pd.DataFrame(
    {
        "row_id": ["r1", "r2", "r3"],
        "account_id": [1, 1, 2],
        "prediction_time": pd.to_datetime(
            ["2026-01-10", "2026-01-15", "2026-01-12"],
            utc=True,
        ),
    }
)

# merge_asof requires the time key to be globally sorted. The "by" key ensures
# that a row can use only events belonging to the same account.
joined = pd.merge_asof(
    prediction_rows.sort_values("prediction_time"),
    feature_events.sort_values("event_time"),
    left_on="prediction_time",
    right_on="event_time",
    by="account_id",
    direction="backward",
    allow_exact_matches=True,
)

assert (joined["event_time"] <= joined["prediction_time"]).all()
print(joined[["row_id", "account_id", "prediction_time", "event_time", "balance"]])
```

</details>

#### **Training-Serving Consistency**

Training-serving skew is a difference between the feature or prediction process used offline and the process used online. It has several forms:

- **logic skew:** two implementations compute different transformations;
- **temporal skew:** training sees complete backfilled history while serving sees partial recent data;
- **source skew:** offline and online paths read from different systems;
- **default skew:** missing values, unknown categories, or clipping are handled differently;
- **version skew:** a model is paired with the wrong feature schema or vocabulary;
- **population skew:** the serving population differs from the training sample.

A feature store can help centralize definitions, historical retrieval, online materialization, and metadata. It does not automatically guarantee correctness. Point-in-time joins, event timestamps, transformation ownership, freshness, and monitoring still need explicit design.

<div class="diagram-scroll wide-diagram">

![Feast architecture separates transformation, registration, storage, and online or offline feature serving.](assets/feast-feature-store-architecture-official.png){fig-alt="Official Feast architecture diagram showing request, stream, and batch sources transformed and registered for online inference and offline training."}

</div>

*Image source: [Feast official repository architecture](https://github.com/feast-dev/feast#-architecture), [Apache License 2.0](https://github.com/feast-dev/feast/blob/master/LICENSE).*

The strongest consistency pattern is to define the transformation once, apply it to raw values in both paths, log a sample of served feature vectors, and replay them through the offline pipeline. For deterministic transformations, parity should be exact or within a declared numerical tolerance:

$$
\max_j\left|x^{\text{train}}_j-x^{\text{serve}}_j\right|\le \epsilon_j.
$$

For stateful or approximate features, compare timestamps, freshness, coverage, and a distribution of differences rather than expecting exact equality.

<details>
<summary><strong>Python: test one shared feature transformation for offline and online parity</strong></summary>

```python
import hashlib
import json
import numpy as np


FEATURE_VERSION = "customer_features_v3"


def transform_customer(raw: dict) -> dict:
    # This function is deliberately shared by training and serving.
    age = float(np.clip(raw["age"], 18, 100))
    income = max(float(raw.get("annual_income") or 0.0), 0.0)
    debt = max(float(raw.get("debt") or 0.0), 0.0)
    return {
        "age_scaled": (age - 18.0) / 82.0,
        "log_income": float(np.log1p(income)),
        "debt_to_income": debt / max(income, 1.0),
        "is_mobile": float(raw.get("channel") == "mobile"),
    }


def fingerprint(features: dict) -> str:
    canonical = json.dumps(features, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()[:12]


raw_record = {
    "age": 34,
    "annual_income": 72000,
    "debt": 18000,
    "channel": "mobile",
}

offline_features = transform_customer(raw_record)
served_log_features = transform_customer(raw_record)

differences = {
    name: abs(offline_features[name] - served_log_features[name])
    for name in offline_features
}
assert max(differences.values()) <= 1e-12

print("feature_version", FEATURE_VERSION)
print("fingerprint", fingerprint(offline_features))
print("maximum_absolute_difference", max(differences.values()))
```

</details>

**Comparison.** Batch pipelines favor replayability and throughput; streaming pipelines favor freshness but require event-time state. Feature stores coordinate definitions and retrieval; they do not remove temporal reasoning. Data contracts catch structural violations, point-in-time joins prevent historical leakage, and parity tests detect divergence between training and serving.


### **Reproducible Training and Experiment Tracking**

An experiment is reproducible only when another run can recover the relevant inputs, procedure, and comparison. Saving the final model is insufficient: the same model class trained on a changed split, dependency version, preprocessing rule, or random seed is a different experiment.

Useful terms are:

- **repeatability:** the same team reruns the same code and setup and obtains materially equivalent results;
- **reproducibility:** another person can use the documented artifacts and procedure to obtain materially equivalent results;
- **replicability:** an independent implementation and study support the same scientific claim.

Exact bitwise equality is not always possible on parallel hardware or nondeterministic accelerators. The required equivalence should therefore be declared. It might be an identical model hash, predictions within a tolerance, a metric within a confidence interval, or the same qualitative ordering among methods.

#### **Data, Code, Environment, and Model Versioning**

A run should identify a complete input tuple:

$$
r=
\bigl(
v_{\text{data}},
v_{\text{code}},
v_{\text{environment}},
c_{\text{config}},
s_{\text{seed}},
h_{\text{hardware}}
\bigr).
$$

- **Data version** includes source snapshots, query text, extraction time, exclusions, split assignments, and hashes.
- **Code version** identifies the commit and records whether the working tree contained uncommitted changes.
- **Environment version** captures the language, libraries, system packages, container image, and relevant drivers.
- **Configuration** stores every parameter that changes preprocessing, training, evaluation, and selection.
- **Seed** controls pseudorandom operations where the implementation honors it.
- **Hardware and execution mode** matter when numerical kernels, precision, parallel order, or device-specific behavior affect results.

A seed is not a reproducibility strategy. It does not freeze input row order, dependency behavior, nondeterministic kernels, hidden global state, or a remotely updated dataset. Multiple independent seeds are often needed to estimate variability rather than conceal it.

<div class="diagram-scroll">

![Experiment lineage connects versioned inputs to runs, artifacts, and decisions.](assets/experiment-lineage.svg){fig-alt="Versioned data, code, environment, and configuration flow into a run, artifacts, and a decision record."}

</div>

<details>
<summary><strong>Python: create a deterministic experiment manifest with content hashes</strong></summary>

```python
import hashlib
import json
import platform
import sys
import numpy as np
import sklearn


def sha256_bytes(value: bytes) -> str:
    return hashlib.sha256(value).hexdigest()


rng = np.random.default_rng(20)
X = rng.normal(size=(120, 5)).astype("float64")
y = (X[:, 0] + 0.4 * X[:, 1] > 0).astype("int8")

config = {
    "model": "logistic_regression",
    "regularization_C": 0.5,
    "split_seed": 20,
    "training_seed": 20,
    "feature_version": "customer_features_v3",
}

manifest = {
    "dataset_sha256": sha256_bytes(X.tobytes() + y.tobytes()),
    "config_sha256": sha256_bytes(
        json.dumps(config, sort_keys=True).encode("utf-8")
    ),
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "platform": platform.platform(),
    # In a real run these values come from version control and the image registry.
    "code_commit": "9d3c02f",
    "container_digest": "sha256:example-image-digest",
}

run_id = sha256_bytes(
    json.dumps(manifest, sort_keys=True).encode("utf-8")
)[:16]

print("run_id", run_id)
print(json.dumps(manifest, indent=2, sort_keys=True))
```

</details>

#### **Seeds, Configurations, and Artifact Management**

Experiment tracking separates **metadata** from **artifacts**:

- metadata are small searchable values such as parameters, metrics, tags, timestamps, status, dataset identifiers, and parent run;
- artifacts are larger files such as model weights, predictions, plots, serialized preprocessors, environment locks, and reports.

The distinction allows a tracking database to answer “which run used feature version 3 and achieved recall above 0.8?” while an artifact store holds the actual model and prediction files. A model registry adds named versions, lineage, review state, aliases, and deployment references. It should not be treated as an approval system by itself; release criteria and accountable sign-off remain explicit.

<div class="diagram-scroll wide-diagram">

![MLflow tracking can evolve from local files to a shared tracking server, metadata database, and artifact store.](assets/mlflow-tracking-architecture-official.png){fig-alt="Official MLflow diagram compares local tracking, local tracking with separate stores, and remote team tracking with a server, database, and cloud artifact store."}

</div>

*Image source: [MLflow Architecture Overview](https://mlflow.org/docs/latest/self-hosting/architecture/overview/), from the [Apache-2.0-licensed MLflow project](https://github.com/mlflow/mlflow/blob/master/LICENSE.txt).*

Every logged metric needs context. `accuracy=0.91` is not interpretable without the dataset and split, population, threshold, model version, metric implementation, and uncertainty. A robust run record commonly stores:

| Artifact | Why it matters |
|---|---|
| Split assignment or stable row identifiers | Verifies that candidates used the same evaluation population |
| Raw predictions and labels | Allows metrics, thresholds, slices, and uncertainty to be recomputed |
| Fitted preprocessing object | Prevents an incompatible transformation at serving time |
| Configuration and environment lock | Reconstructs the training procedure |
| Training curves and resource logs | Diagnoses convergence and compute cost |
| Model signature and schema | Validates serving inputs and outputs |
| Decision record | Explains why a run was promoted, rejected, or superseded |

<details>
<summary><strong>Python: implement a minimal run tracker with immutable run records</strong></summary>

```python
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
import hashlib
import json


@dataclass(frozen=True)
class RunRecord:
    run_id: str
    created_at: str
    parameters: dict
    metrics: dict
    artifacts: dict
    tags: dict = field(default_factory=dict)


def create_run(parameters, metrics, artifacts, tags=None):
    payload = {
        "parameters": parameters,
        "metrics": metrics,
        "artifacts": artifacts,
        "tags": tags or {},
    }
    run_id = hashlib.sha256(
        json.dumps(payload, sort_keys=True).encode("utf-8")
    ).hexdigest()[:12]
    return RunRecord(
        run_id=run_id,
        created_at=datetime.now(timezone.utc).isoformat(),
        parameters=dict(parameters),
        metrics=dict(metrics),
        artifacts=dict(artifacts),
        tags=dict(tags or {}),
    )


registry = {}
record = create_run(
    parameters={"model": "logistic", "C": 0.5, "seed": 20},
    metrics={"validation_log_loss": 0.384, "validation_recall_at_500": 0.812},
    artifacts={
        "model": "artifacts/model.pkl",
        "predictions": "artifacts/validation_predictions.parquet",
        "environment": "artifacts/requirements.lock",
    },
    tags={"dataset": "events@2026-04-01", "code_commit": "9d3c02f"},
)
registry[record.run_id] = record

print(json.dumps(asdict(registry[record.run_id]), indent=2, sort_keys=True))
```

</details>

Real platforms add transactions, authentication, remote storage, lineage queries, and lifecycle policies. The minimal example reveals the conceptual contract: parameters, metrics, artifacts, and provenance form one immutable run record. A mutable spreadsheet row that is overwritten by the next experiment cannot provide the same audit trail.

**Comparison.** Version control answers what changed; environment capture answers what executed; experiment tracking answers what happened; artifact storage preserves the evidence; a model registry identifies released versions. None of these alone guarantees scientific validity, but together they make invalid or irreproducible claims much easier to detect.


### **Deployment and Serving**

Deployment converts a reviewed experiment artifact into a versioned service or batch job that can receive valid inputs, produce defined outputs, meet operational objectives, and be reversed. It is not equivalent to copying a serialized estimator. The deployable unit usually contains:

- feature schema, order, types, vocabulary, and missing-value policy;
- preprocessing and post-processing code;
- model parameters and inference implementation;
- input and output signatures;
- dependency and runtime versions;
- health checks, telemetry, and version identifiers;
- resource requests and concurrency limits;
- fallback and rollback instructions.

Before release, the package should pass **unit tests** for transformations, **contract tests** for schemas, **integration tests** for the complete request path, **replay tests** against historical requests, **load tests** for latency and saturation, and **behavior tests** for invariants and critical slices. Offline accuracy cannot reveal a missing feature, incompatible category encoder, thread-safety bug, or overloaded downstream store.

#### **Batch, Online, and Edge Inference**

Serving mode follows the deadline and location of the decision.

<div class="diagram-scroll">

![Batch, online, and edge serving exchange freshness, latency, and resource constraints.](assets/serving-modes.svg){fig-alt="Three panels compare batch inference, low-latency online inference, and resource-constrained edge inference."}

</div>

| Mode | Best fit | Main advantages | Main risks |
|---|---|---|---|
| Batch | Scores are consumed on a schedule | High throughput, efficient vectorization, easy replay | Stale predictions and large failed jobs |
| Online | The action waits for a request-time score | Fresh context and immediate decisions | Tail latency, availability, concurrency, online feature failures |
| Edge | Data or action must stay on a device or local site | Privacy boundary, offline operation, low network dependence | Tight memory/energy, hardware diversity, slow fleet updates |

Static training can coexist with dynamic online inference; dynamic training can coexist with batch inference. **Training cadence** and **serving cadence** are independent design choices. Retraining every hour is wasteful if the underlying relation changes monthly, while an annually trained model can still need millisecond request-time predictions.

#### **Latency, Throughput, Memory, and Cost**

End-to-end latency is a sum of dependent stages:

$$
T_{\text{request}}
=T_{\text{network}}
+T_{\text{feature}}
+T_{\text{queue}}
+T_{\text{model}}
+T_{\text{post}}
+T_{\text{downstream}}.
$$

The p99 of the sum is not generally the sum of component p99 values, because stages can be correlated and different requests occupy each tail. Measure the complete path under realistic concurrency. Mean latency is especially misleading when user-visible timeouts are caused by the tail.

Throughput $\lambda$ is the number of requests handled per unit time. When a stable queue has mean time in the system $W$, Little's law gives the average number of in-flight requests:

$$
L=\lambda W.
$$

If traffic arrives faster than service capacity, queueing delay grows sharply. Batching increases hardware utilization but may add waiting time. Caching reduces computation but requires a key, expiration policy, invalidation logic, and an analysis of stale results.

Memory includes model parameters, runtime overhead, feature buffers, request batches, caches, and duplicated worker state. Cost should be normalized to a useful unit, such as cost per thousand predictions, per accepted case, or per correct decision, rather than reported as a monthly total without workload.

<details>
<summary><strong>Python: decompose latency and test an end-to-end service objective</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(20)
n_requests = 100_000

# Shared load makes stages correlated: busy requests wait longer for both
# feature retrieval and model execution.
load = rng.lognormal(mean=-2.0, sigma=0.7, size=n_requests)
network = rng.gamma(2.0, 1.2, n_requests)
feature = rng.gamma(2.0, 2.0, n_requests) + 18.0 * load
queue = rng.exponential(1.0 + 12.0 * load)
model = rng.gamma(3.0, 1.4, n_requests) + 10.0 * load
post = rng.gamma(2.0, 0.5, n_requests)
end_to_end = network + feature + queue + model + post


def percentiles(values):
    return {
        "p50": round(float(np.quantile(values, 0.50)), 2),
        "p95": round(float(np.quantile(values, 0.95)), 2),
        "p99": round(float(np.quantile(values, 0.99)), 2),
    }


for name, values in {
    "network": network,
    "feature": feature,
    "queue": queue,
    "model": model,
    "post": post,
    "end_to_end": end_to_end,
}.items():
    print(name, percentiles(values))

latency_slo_ms = 80.0
print("p99_slo_pass", np.quantile(end_to_end, 0.99) <= latency_slo_ms)
print(
    "incorrect_sum_of_component_p99",
    round(sum(np.quantile(v, 0.99) for v in [network, feature, queue, model, post]), 2),
)
```

</details>

#### **Packaging, Safe Rollout, and Rollback**

Release should increase exposure gradually while preserving a known-good path.

<div class="diagram-scroll">

![Safe deployment progresses through offline, shadow, canary, ramp, and operation gates.](assets/safe-rollout-stages.svg){fig-alt="Five deployment stages are connected by evidence gates, with a rollback loop returning traffic to a known-good version."}

</div>

- **Shadow deployment** sends a copy of production requests to the candidate but does not let its output affect users. It detects schema, latency, numerical, and prediction-distribution differences on realistic traffic.
- **Canary deployment** sends a small randomized traffic fraction to the candidate. It tests operational and outcome guardrails with limited exposure.
- **A/B testing** randomizes eligible units between policies to estimate causal effects on a product or decision outcome. The unit of randomization must prevent interference and repeated-user contamination.
- **Blue-green deployment** keeps two complete environments and switches traffic between them, making rollback fast at higher infrastructure cost.

Promotion criteria should be declared before observing canary results. A rollback trigger might combine error rate, p99 latency, prediction coverage, and a high-severity slice metric. The fallback may be a previous model, a deterministic rule, cached score, abstention, or human review. “Turn the service off” is not an adequate fallback when the downstream application requires a response.

<details>
<summary><strong>Python: evaluate a canary against operational guardrails</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(20)
n_control = 60_000
n_canary = 6_000

# Simulated request telemetry. The canary improves outcome utility slightly
# but has a heavier latency tail and a higher error probability.
control_latency = rng.lognormal(mean=3.35, sigma=0.34, size=n_control)
canary_latency = rng.lognormal(mean=3.45, sigma=0.43, size=n_canary)
control_errors = rng.binomial(1, 0.0012, n_control)
canary_errors = rng.binomial(1, 0.0030, n_canary)
control_utility = rng.normal(1.00, 0.50, n_control)
canary_utility = rng.normal(1.06, 0.50, n_canary)

report = {
    "control_p99_ms": float(np.quantile(control_latency, 0.99)),
    "canary_p99_ms": float(np.quantile(canary_latency, 0.99)),
    "control_error_rate": float(control_errors.mean()),
    "canary_error_rate": float(canary_errors.mean()),
    "utility_lift": float(canary_utility.mean() - control_utility.mean()),
}

guardrails = {
    "p99_latency_under_80ms": report["canary_p99_ms"] <= 80.0,
    "error_rate_under_0.002": report["canary_error_rate"] <= 0.002,
    "positive_utility_lift": report["utility_lift"] > 0,
}

print({key: round(value, 5) for key, value in report.items()})
print("guardrails", guardrails)
print("decision", "PROMOTE" if all(guardrails.values()) else "ROLL_BACK")
```

</details>

The candidate is rolled back even if its average utility appears better, because a predefined error-rate guardrail fails. With real data, uncertainty, sequential monitoring, minimum sample size, seasonality, and multiple metrics require more careful experimental design. The key principle remains: exposure is reversible and promotion is evidence-driven.

**Comparison.** Batch serving optimizes throughput, online serving optimizes request-time freshness, and edge serving optimizes locality. Shadowing tests compatibility without impact; canaries limit operational risk; A/B tests estimate policy effects. Packaging defines what is executed, while rollback defines how the system fails safely.


### **Monitoring and Maintenance**

Monitoring is the continuous process of collecting evidence, detecting actionable deviations, diagnosing likely causes, and triggering an owned response. A collection of dashboards is observability, not yet an operational control. Every alert needs:

1. a named signal and aggregation window;
2. a threshold or statistical rule with known false-alarm behavior;
3. an owner and severity;
4. a diagnostic playbook;
5. an action such as investigate, degrade gracefully, roll back, retrain, or stop;
6. a record of the incident and its resolution.

Monitoring must cover the complete chain from infrastructure to outcomes.

<div class="diagram-scroll">

![Monitoring spans system, data, model, and decision layers.](assets/monitoring-layers.svg){fig-alt="Four panels list observability signals for system infrastructure, data, model behavior, and downstream decisions."}

</div>

#### **Data Quality, Drift, and Performance Decay**

**Data quality monitoring** checks whether inputs satisfy their contract now. Typical signals include schema changes, missingness, duplicates, invalid categories, range violations, freshness, volume, entity coverage, and feature computation failures.

**Distribution monitoring** compares a current population $Q(X)$ with a reference population $P(X)$. It can detect covariate movement but cannot by itself determine whether the model became worse. Performance depends on the joint relation among $X$, $Y$, and the decision policy. A feature can drift while the prediction remains valid, or accuracy can collapse while marginal feature distributions look stable.

Common univariate drift statistics include:

- Kolmogorov-Smirnov distance for continuous empirical distributions;
- Jensen-Shannon divergence for discrete distributions;
- Wasserstein distance when the geometry of values matters;
- Population Stability Index (PSI), commonly used with fixed reference bins;
- changes in missingness, category frequency, quantiles, or out-of-range rate.

For reference bin proportions $p_b$ and current proportions $q_b$, PSI is

$$
\operatorname{PSI}(P,Q)
=
\sum_{b=1}^{B}
(q_b-p_b)\log\frac{q_b}{p_b}.
$$

Smoothing is needed when a bin is empty. PSI has no universal operational threshold: its value changes with binning, sample size, smoothing, and feature distribution. It should be calibrated against historical benign periods and known incidents, then combined with feature importance, model behavior, and slice-level diagnostics.

<details>
<summary><strong>Python: compute PSI with fixed reference bins and inspect its sensitivity</strong></summary>

```python
import numpy as np


def population_stability_index(reference, current, bins=10, epsilon=1e-6):
    # Quantile edges are learned only from the reference period and then frozen.
    edges = np.quantile(reference, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    edges = np.unique(edges)

    reference_counts, _ = np.histogram(reference, bins=edges)
    current_counts, _ = np.histogram(current, bins=edges)

    p = reference_counts / reference_counts.sum()
    q = current_counts / current_counts.sum()
    p = np.clip(p, epsilon, None)
    q = np.clip(q, epsilon, None)
    contributions = (q - p) * np.log(q / p)
    return float(contributions.sum()), edges, contributions


rng = np.random.default_rng(20)
reference = rng.normal(loc=0.0, scale=1.0, size=20_000)
benign_current = rng.normal(loc=0.05, scale=1.0, size=20_000)
shifted_current = rng.normal(loc=0.70, scale=1.25, size=20_000)

for name, sample in {
    "benign": benign_current,
    "shifted": shifted_current,
}.items():
    psi, edges, contributions = population_stability_index(reference, sample)
    print(name, "psi", round(psi, 4), "largest_bin_contribution", round(contributions.max(), 4))
```

</details>

Monitor both raw inputs and model-facing features. A healthy raw source can feed a broken transformation; a feature distribution can remain stable while the mapping from feature to outcome changes. Prediction monitoring should include score distribution, predicted class rate, confidence, calibration, abstention, coverage, and critical slices. Compare versions on the same logged traffic when possible.

#### **Delayed Labels and Production Performance**

Many outcomes arrive after the prediction: chargebacks, loan defaults, subscription renewal, recovery, disease progression, or long-term engagement. Production evaluation therefore has two clocks.

<div class="diagram-scroll">

![Delayed-label evaluation joins mature outcomes back to logged predictions.](assets/delayed-label-clocks.svg){fig-alt="Prediction time, delayed outcome maturity, evaluation, and operational action appear as four connected stages."}

</div>

Each prediction log should contain a stable example or entity key, event time, model version, feature version, score, decision, eligibility, and relevant slice attributes. When labels mature, evaluation should use a fixed cohort:

$$
\mathcal{C}_t
=
\left\{
i:
t_i\le t-h
\;\land\;
\text{outcome window for }i\text{ is complete}
\right\},
$$

where $h$ is the label horizon. Mixing mature old cases with immature recent cases creates censoring bias. Report **label coverage** and reasons for missing labels. If actions affect observation, such as reviewing only high-risk cases, the labeled subset is selective and ordinary accuracy estimates may be biased.

<details>
<summary><strong>Python: join delayed outcomes and evaluate only a mature cohort</strong></summary>

```python
import pandas as pd
from sklearn.metrics import log_loss, roc_auc_score

predictions = pd.DataFrame(
    {
        "case_id": ["a", "b", "c", "d", "e", "f"],
        "prediction_time": pd.to_datetime(
            ["2026-01-01", "2026-01-05", "2026-01-20", "2026-02-01", "2026-02-15", "2026-03-10"],
            utc=True,
        ),
        "model_version": ["v7", "v7", "v7", "v8", "v8", "v8"],
        "score": [0.80, 0.25, 0.60, 0.35, 0.72, 0.55],
    }
)

outcomes = pd.DataFrame(
    {
        "case_id": ["a", "b", "c", "d", "e"],
        "outcome_time": pd.to_datetime(
            ["2026-01-12", "2026-01-25", "2026-02-11", "2026-02-20", "2026-03-12"],
            utc=True,
        ),
        "label": [1, 0, 1, 0, 1],
    }
)

as_of = pd.Timestamp("2026-04-01", tz="UTC")
label_horizon = pd.Timedelta(days=30)
joined = predictions.merge(outcomes, on="case_id", how="left")
joined["mature"] = joined["prediction_time"] + label_horizon <= as_of

mature = joined[joined["mature"]]
labeled = mature.dropna(subset=["label"]).copy()
coverage = len(labeled) / len(mature)

print("mature_cases", len(mature), "label_coverage", round(coverage, 3))
print("auc", round(roc_auc_score(labeled["label"], labeled["score"]), 3))
print("log_loss", round(log_loss(labeled["label"], labeled["score"]), 3))
print(labeled.groupby("model_version").size().rename("evaluated_cases"))
```

</details>

Before labels mature, input and score drift are useful **proxies** for diagnosis, but they are not evidence that predictive performance changed. A rapid proxy alert can trigger investigation; promotion, rollback, or retraining should use the strongest available evidence and acknowledge delay.

#### **Service-Level Objectives and Incident Response**

A **service-level indicator (SLI)** is a measured quantity, such as successful prediction responses within 80 ms. A **service-level objective (SLO)** is a target for that indicator over a window, such as 99.9% over 30 days. The permitted failure fraction is the **error budget**:

$$
\text{error budget}
=
N\left(1-\text{SLO target}\right)
$$

for $N$ eligible requests. Error budgets turn vague reliability goals into trade-offs. Consuming the budget rapidly can pause risky releases and prioritize reliability work. ML services often need several SLOs: availability, latency, valid-feature coverage, prediction coverage, and freshness.

Incidents should be triaged by layer:

1. **System:** timeouts, resource saturation, dependency failure, deployment error.
2. **Data:** schema, volume, freshness, invalid values, source outage.
3. **Model:** score collapse, calibration shift, unstable slice, wrong version.
4. **Decision:** workload overload, changed policy, adverse feedback, unexpected harm.

The response should preserve evidence. Log the timeline, affected versions and cohorts, automated actions, manual decisions, user impact, root cause, and prevention work. Deleting the failed artifact or rewriting the original experiment destroys the learning value of the incident.

<details>
<summary><strong>Python: calculate error-budget burn and map it to an operational action</strong></summary>

```python
requests_30d = 12_000_000
slo_target = 0.999
allowed_failures = requests_30d * (1.0 - slo_target)

# Four hours into a 30-day window, a dependency incident has caused failures.
elapsed_hours = 4
window_hours = 30 * 24
observed_failures = 1850

expected_budget_by_now = allowed_failures * elapsed_hours / window_hours
burn_rate = observed_failures / expected_budget_by_now
remaining_budget = allowed_failures - observed_failures

if burn_rate >= 14.4:
    action = "PAGE_AND_ROLL_BACK"
elif burn_rate >= 6.0:
    action = "PAGE_AND_FREEZE_RELEASES"
elif burn_rate >= 2.0:
    action = "INVESTIGATE"
else:
    action = "CONTINUE_MONITORING"

print("allowed_failures_30d", round(allowed_failures))
print("observed_failures", observed_failures)
print("remaining_budget", round(remaining_budget))
print("burn_rate", round(burn_rate, 2))
print("action", action)
```

</details>

**Comparison.** Contract checks detect invalid inputs; drift metrics detect population movement; prediction telemetry detects changed model behavior; mature labels estimate predictive quality; decision outcomes estimate real value. SLOs govern service reliability, while incident response turns failures into owned corrective action.


### **Feedback Loops and Retraining Policies**

Once predictions affect decisions, the model becomes part of the data-generating process. A recommender changes exposure, a fraud model changes attacker behavior, a screening model changes which cases receive definitive tests, and a risk score changes which users receive an intervention. The next dataset is therefore conditioned on earlier policies.

<div class="diagram-scroll">

![Model scores can change decisions, observations, and future training data.](assets/feedback-loop-dynamics.svg){fig-alt="A loop connects model score, decision, observed data, and retraining, showing that predictions alter future samples."}

</div>

Several feedback mechanisms should be distinguished:

- **selective labels:** outcomes are observed only for units receiving an action, such as loans that were approved;
- **exposure bias:** a ranking system observes engagement mainly for items it displayed;
- **self-fulfilling prediction:** intervention makes the predicted outcome more likely;
- **self-defeating prediction:** intervention prevents the predicted outcome, making the model appear wrong;
- **behavioral adaptation:** users, markets, or adversaries respond strategically;
- **population composition:** the policy changes who enters or remains in the system;
- **measurement feedback:** the model changes how labels or features are recorded.

Suppose a historical policy approves an application when score $S$ exceeds threshold $\tau$. The repayment label $Y$ is then observed mainly when

$$
A=\mathbb{1}[S\ge\tau]=1.
$$

Training on $P(Y\mid X,A=1)$ does not generally recover $P(Y\mid X)$ for rejected applications. Missing labels are not missing at random because the decision depended on risk-related features. Simply treating unobserved outcomes as negative creates a stronger bias.

Mitigation depends on the mechanism:

- preserve a randomized exploration sample when ethically and operationally possible;
- use randomized or quasi-experimental data to estimate policy effects;
- collect independent audits or delayed labels outside the decision path;
- model observation propensity and use weighting only when its assumptions are defensible;
- log eligibility, exposure, action, override, and observation status;
- evaluate on policy-invariant or externally labeled cohorts;
- include humans for escalation, but measure human disagreement and automation bias.

<details>
<summary><strong>Python: demonstrate selective-label bias created by a historical policy</strong></summary>

```python
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(20)
def generate_population(size):
    income = rng.normal(size=size)
    risk = rng.normal(size=size)
    # A nonlinear interaction is especially important in the low-income region
    # that the historical policy rarely labels.
    logit = (
        -0.3 + 0.9 * income - 1.2 * risk
        + 2.2 * (income < -0.6) * risk
    )
    probability = 1 / (1 + np.exp(-logit))
    label = rng.binomial(1, probability)
    return np.column_stack([income, risk]), label


X_train, y_train = generate_population(60_000)
X_test, y_test = generate_population(30_000)

# Historical approval depends on a noisy score related to the outcome. Labels
# from rejected training cases are hidden, but the complete test population is
# available in this simulation for auditing.
historical_score = (
    0.9 * X_train[:, 0] - 0.9 * X_train[:, 1]
    + rng.normal(0, 0.45, len(X_train))
)
approved = historical_score > 0.2

observed_model = HistGradientBoostingClassifier(
    max_iter=120, max_leaf_nodes=15, random_state=20
).fit(X_train[approved], y_train[approved])
oracle_model = HistGradientBoostingClassifier(
    max_iter=120, max_leaf_nodes=15, random_state=20
).fit(X_train, y_train)

observed_auc_all = roc_auc_score(
    y_test, observed_model.predict_proba(X_test)[:, 1]
)
oracle_auc_all = roc_auc_score(
    y_test, oracle_model.predict_proba(X_test)[:, 1]
)

print("training_label_coverage", round(approved.mean(), 3))
print("approved_positive_rate", round(y_train[approved].mean(), 3))
print("rejected_positive_rate_hidden_from_training", round(y_train[~approved].mean(), 3))
print("observed_only_model_auc_on_full_population", round(observed_auc_all, 4))
print("oracle_model_auc_on_full_population", round(oracle_auc_all, 4))
```

</details>

The code can inspect rejected outcomes because it is a simulation. A real system often cannot, which is exactly why high validation performance on approved cases does not establish quality on all applicants.

#### **Retraining Triggers**

Retraining is a controlled change to a deployed decision system, not routine housekeeping. Common policies are:

- **scheduled:** retrain daily, weekly, or monthly;
- **data-volume based:** retrain after enough new mature examples arrive;
- **drift triggered:** retrain after persistent, material input or score movement;
- **performance triggered:** retrain after statistically credible degradation on mature labels;
- **event triggered:** retrain after a product, policy, sensor, vocabulary, or market change;
- **human approved:** an owner reviews evidence and initiates the run.

Each trigger has limitations. Scheduled retraining can replace a healthy model with a noisy one. Drift does not prove that a new model will be better. Performance alerts arrive late when labels are delayed. Event triggers depend on organizational communication.

A robust policy separates **trigger**, **candidate training**, **validation**, and **promotion**:

$$
\text{signal}
\rightarrow
\text{train candidate}
\rightarrow
\text{compare with incumbent}
\rightarrow
\text{release gates}
\rightarrow
\text{monitor}.
$$

Retraining never implies automatic promotion. The incumbent remains the baseline and fallback. Use hysteresis, persistence windows, and cooldowns to prevent oscillation:

- trigger only if degradation exceeds $\tau_{\text{high}}$ for $k$ windows;
- clear the incident only when it falls below $\tau_{\text{low}}<\tau_{\text{high}}$;
- wait a cooldown interval after a release or failed training job;
- require enough mature labels and uncertainty narrow enough to decide.

<details>
<summary><strong>Python: implement a stateful retraining trigger with persistence and cooldown</strong></summary>

```python
from dataclasses import dataclass


@dataclass
class RetrainingPolicy:
    high_threshold: float = 0.045
    low_threshold: float = 0.025
    persistence_windows: int = 3
    cooldown_windows: int = 4
    consecutive_high: int = 0
    cooldown_remaining: int = 0
    incident_open: bool = False

    def update(self, performance_drop: float, mature_labels: int) -> str:
        if self.cooldown_remaining > 0:
            self.cooldown_remaining -= 1
            return "COOLDOWN"

        if mature_labels < 1000:
            return "WAIT_FOR_LABELS"

        if performance_drop >= self.high_threshold:
            self.consecutive_high += 1
        else:
            self.consecutive_high = 0

        if self.incident_open and performance_drop <= self.low_threshold:
            self.incident_open = False
            return "RECOVERED"

        if self.consecutive_high >= self.persistence_windows:
            self.incident_open = True
            self.consecutive_high = 0
            self.cooldown_remaining = self.cooldown_windows
            return "TRAIN_CANDIDATE"

        return "MONITOR"


policy = RetrainingPolicy()
windows = [
    (0.018, 1400), (0.052, 1500), (0.049, 1550), (0.051, 1600),
    (0.060, 1700), (0.040, 1800), (0.020, 1900), (0.055, 2000),
]

for index, (drop, labels) in enumerate(windows, start=1):
    print(index, drop, policy.update(drop, labels))
```

</details>

After `TRAIN_CANDIDATE`, a pipeline should snapshot eligible data, produce a versioned run, compare against the incumbent on unchanged release criteria, and either reject or safely deploy it. The trigger itself is evidence that investigation is warranted, not evidence that the newest available data or architecture will solve the problem.

**Comparison.** Ordinary drift assumes the world changes independently of the model; feedback loops recognize that the policy changes observation and behavior. Retraining restores the experimentation cycle, but promotion still requires baseline comparison, uncertainty, constraints, and reversible deployment.


### **Scalability and Efficient Learning**

Scalability means that the training and serving process remains feasible as data volume, feature dimension, model size, request rate, or team usage grows. It is not synonymous with distributed training. The first question is which resource actually binds:

- CPU or accelerator compute;
- host or device memory;
- storage capacity and read bandwidth;
- network communication;
- feature retrieval latency;
- request concurrency;
- annotation or human-review capacity;
- experiment turnaround time;
- energy or monetary budget.

Measure before optimizing. A profiler can reveal that feature joins dominate a training job while GPU utilization remains low, or that an online model is fast but remote feature retrieval dominates p99 latency. Scaling the wrong component increases cost without increasing throughput.

<div class="diagram-scroll">

![Resource-aware model selection chooses a feasible point on the utility-cost frontier.](assets/resource-aware-frontier.svg){fig-alt="A curve shows diminishing utility gains as compute, latency, and memory cost increase, together with utility and resource constraints."}

</div>

#### **Parallelism, Approximation, and Resource-Aware Modeling**

Common scaling strategies operate at different layers:

- **Data parallelism:** workers process different examples and aggregate gradients or statistics. Communication and synchronization can dominate.
- **Model parallelism:** parameters or layers are divided across devices when one device cannot hold the model.
- **Pipeline parallelism:** model stages process different micro-batches concurrently, trading utilization against scheduling complexity.
- **Vectorization and batching:** use efficient kernels and amortize overhead without changing the model.
- **Sampling and sketching:** reduce rows, negatives, categories, or sufficient statistics with controlled approximation.
- **Sparse representations:** avoid storing or multiplying zeros.
- **Quantization:** represent weights or activations with fewer bits, requiring accuracy and hardware checks.
- **Pruning:** remove weights, branches, features, or experts; unstructured sparsity helps only when the runtime exploits it.
- **Distillation:** train a smaller student to approximate a larger teacher.
- **Caching and precomputation:** exchange freshness and storage for lower request-time compute.

For candidate $m$, define a vector

$$
z_m=
\bigl(
\text{utility}_m,
-\text{latency}_m,
-\text{memory}_m,
-\text{cost}_m
\bigr).
$$

Candidate $a$ **dominates** $b$ if it is at least as good in every dimension and strictly better in one. Dominated models need no subjective weighting: another candidate is better under all recorded criteria. The remaining Pareto frontier exposes real trade-offs for the decision owner.

<details>
<summary><strong>Python: remove dominated models and apply hard resource constraints</strong></summary>

```python
candidates = [
    {"name": "linear", "utility": 0.742, "p99_ms": 6, "memory_mb": 8, "cost": 0.02},
    {"name": "small_tree", "utility": 0.781, "p99_ms": 11, "memory_mb": 28, "cost": 0.05},
    {"name": "boosted", "utility": 0.824, "p99_ms": 38, "memory_mb": 180, "cost": 0.35},
    {"name": "ensemble", "utility": 0.831, "p99_ms": 96, "memory_mb": 720, "cost": 1.80},
    {"name": "compressed", "utility": 0.816, "p99_ms": 22, "memory_mb": 95, "cost": 0.18},
    {"name": "legacy", "utility": 0.760, "p99_ms": 45, "memory_mb": 240, "cost": 0.60},
]


def dominates(a, b):
    no_worse = (
        a["utility"] >= b["utility"]
        and a["p99_ms"] <= b["p99_ms"]
        and a["memory_mb"] <= b["memory_mb"]
        and a["cost"] <= b["cost"]
    )
    strictly_better = (
        a["utility"] > b["utility"]
        or a["p99_ms"] < b["p99_ms"]
        or a["memory_mb"] < b["memory_mb"]
        or a["cost"] < b["cost"]
    )
    return no_worse and strictly_better


frontier = [
    candidate
    for candidate in candidates
    if not any(dominates(other, candidate) for other in candidates if other is not candidate)
]
feasible = [
    candidate
    for candidate in frontier
    if candidate["p99_ms"] <= 50
    and candidate["memory_mb"] <= 256
    and candidate["cost"] <= 0.50
]

print("pareto_frontier", [candidate["name"] for candidate in frontier])
print("feasible_frontier", [candidate["name"] for candidate in feasible])
print("best_feasible_utility", max(feasible, key=lambda row: row["utility"]))
```

</details>

The “legacy” model is removed because it is worse than another candidate in every recorded dimension. The ensemble remains non-dominated but violates the hard serving contract. Compression is worthwhile only if the measured quality-resource trade-off is better than directly training a smaller model.

### **AutoML and Automated Model Selection**

Automated machine learning searches over a defined pipeline space. Depending on scope, the search variables can include:

- imputation, encoding, scaling, feature selection, and resampling;
- model family and hyperparameters;
- architecture, augmentation, optimizer, and schedule;
- threshold, calibration, ensemble, and compression choices;
- resource allocation and early stopping.

Automation does not choose the scientific question, prevent leakage, justify the metric, define deployment constraints, or establish external validity. It amplifies the protocol it is given. If preprocessing is fitted before cross-validation, a large search can optimize leakage more efficiently than a small search.

Search strategies include:

- **grid search:** systematic but grows exponentially with dimensions;
- **random search:** often stronger when only a few dimensions matter;
- **Bayesian optimization:** models performance as a function of configurations and selects informative trials;
- **evolutionary or population methods:** mutate and recombine candidates;
- **multi-fidelity methods:** allocate small budgets first and promote promising configurations;
- **bandit schedulers:** stop weak trials early based on intermediate evidence.

In successive halving, start with $n_0$ configurations at resource $r_0$. With reduction factor $\eta>1$, each round keeps approximately $1/\eta$ of the candidates and multiplies resource by $\eta$:

$$
n_k\approx\frac{n_0}{\eta^k},
\qquad
r_k=r_0\eta^k.
$$

Resource may be examples, iterations, trees, epochs, or data resolution. The method assumes low-budget performance is informative about high-budget performance. It can discard slow-starting but ultimately strong configurations, so rankings and learning curves should be audited.

<details>
<summary><strong>Python: implement leakage-safe successive halving over regularization</strong></summary>

```python
import math
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = make_classification(
    n_samples=7000,
    n_features=30,
    n_informative=10,
    n_redundant=5,
    weights=[0.72, 0.28],
    random_state=20,
)
X_train, X_validation, y_train, y_validation = train_test_split(
    X, y, test_size=1500, stratify=y, random_state=20
)

rng = np.random.default_rng(20)
order = rng.permutation(len(X_train))
X_train, y_train = X_train[order], y_train[order]

candidates = [{"C": value} for value in np.logspace(-3, 2, 9)]
resources = [600, 1800, len(X_train)]
reduction_factor = 3

for round_index, resource in enumerate(resources, start=1):
    scored = []
    for config in candidates:
        # Scaling is fitted inside each candidate pipeline on only the allocated
        # training subset; the validation set remains untouched.
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(C=config["C"], max_iter=2000),
        )
        model.fit(X_train[:resource], y_train[:resource])
        probability = model.predict_proba(X_validation)[:, 1]
        scored.append((log_loss(y_validation, probability), config))

    scored.sort(key=lambda item: item[0])
    keep = 1 if round_index == len(resources) else max(
        1, math.ceil(len(scored) / reduction_factor)
    )
    candidates = [config for _, config in scored[:keep]]
    print(
        "round", round_index,
        "resource", resource,
        "best_log_loss", round(scored[0][0], 4),
        "survivors", [round(item["C"], 5) for item in candidates],
    )
```

</details>

The final untouched test set should be evaluated only after the search has selected a pipeline. If AutoML is used to report an unbiased estimate rather than merely choose a model, wrap the entire search inside an outer validation loop or use a separate test set. Search budget, failures, and all attempted configurations are part of the experimental record.

**Comparison.** Systems scaling removes the real bottleneck; model compression changes the quality-resource frontier; distributed execution increases capacity; AutoML allocates experimental budget across a search space. None replaces a valid problem contract or leakage-safe evaluation.


### **Reading and Reproducing Research Papers**

Reading a paper is an exercise in reconstructing a claim and its evidence, not collecting architecture names. A paper may contain theoretical claims, empirical claims, engineering claims, and qualitative observations; each needs a different validation method. Start by rewriting the central statement in a testable form:

> Under population $P$, dataset and split protocol $D$, resource budget $B$, metric $M$, and comparator set $\mathcal{C}$, method $A$ improves quantity $\Delta$ with uncertainty $U$.

If any element is absent, mark it as an assumption rather than silently filling it in. A result on one dataset under one random split is not automatically a claim about a domain, and a gain obtained with a larger pretraining corpus or search budget is not an architecture-only gain.

<div class="diagram-scroll">

![Credible research connects a precise claim to protocol, evidence, challenge tests, and artifacts.](assets/research-evidence-chain.svg){fig-alt="Five stages connect a claim to protocol, evidence, challenge tests, and an executable artifact."}

</div>

#### **Recovering Assumptions and Experimental Protocols**

A structured first reading can use five passes:

1. **Claim pass:** identify the problem, proposed contribution, comparison, and scope.
2. **Method pass:** derive the mathematical objective, data flow, inference rule, and complexity.
3. **Protocol pass:** recover datasets, split unit, preprocessing, hyperparameter search, seeds, compute, selection rule, and metrics.
4. **Evidence pass:** inspect baselines, uncertainty, ablations, sensitivity, negative results, and failure cases.
5. **Artifact pass:** check code, data access, environment, commands, checkpoints, licenses, and discrepancies between paper and implementation.

Create a claim table:

| Claim | Required comparison | Key control | Threat if missing |
|---|---|---|---|
| Better predictive method | Strong tuned baselines under equal data and budget | Same split, metric, preprocessing, and search budget | Gain may come from protocol advantage |
| More data efficient | Learning curve over labeled sample size | Same unlabeled data and pretraining | Hidden supervision advantage |
| More robust | Named shifts and severity levels | Same clean-performance regime | Robustness may reflect lower clean capacity |
| Faster | End-to-end wall time and hardware utilization | Same hardware, precision, batch, and quality target | Kernel-only speedup may not improve workflow |
| More interpretable | Defined audience, target, and fidelity test | Comparable prediction quality | Plausibility mistaken for faithfulness |

Important details often hide in appendices, configuration files, scripts, issue trackers, or data loaders. Recover:

- exact train/validation/test identifiers and whether entities cross splits;
- data exclusions and deduplication;
- checkpoint selection and early stopping;
- preprocessing fitted on which rows;
- number of trials, seeds, and failed runs;
- hardware, numerical precision, and runtime;
- test-set reuse during development;
- whether tables show the best run, mean, median, or an ensemble.

The absence of a detail should reduce confidence proportionally to its ability to change the conclusion.

#### **From-Scratch Implementation and Reference Reproduction**

Different reproduction goals answer different questions:

- **reference rerun:** execute the authors' code and recover reported behavior;
- **clean-room reimplementation:** implement the method from the paper without copying its code;
- **component verification:** test a mathematical or algorithmic component on a controlled synthetic problem;
- **replication:** evaluate the claim on new data, environments, implementations, or teams;
- **extension:** change one assumption and study when the result does or does not generalize.

Begin with the smallest executable case. Verify tensor shapes, loss terms, masking, normalization, boundary conditions, and a tiny overfit test. For probabilistic methods, test normalization and compare against an exact small case. For optimization, inspect gradient checks and learning curves. For data pipelines, hand-audit a few examples from raw source to model input.

A reproduction log should separate:

1. **Paper specification:** what the text and equations state.
2. **Reference behavior:** what the released implementation does.
3. **Your implementation:** choices made to resolve ambiguity.
4. **Observed discrepancy:** numerical or behavioral difference.
5. **Investigation:** hypotheses, controlled tests, and evidence.
6. **Conclusion:** supported scope, unresolved uncertainty, and artifact versions.

Matching one headline number is weaker than matching the learning curve, ablation ordering, qualitative failure modes, and resource profile. Conversely, a numerical mismatch does not automatically falsify a claim if hardware, stochasticity, or data access differs; report the equivalence criterion established before the run.

#### **Ablation, Sensitivity, and Failure Analysis**

An ablation asks whether a component is necessary under the tested protocol. A sensitivity study asks how conclusions change across reasonable parameter, data, or environment choices. A failure analysis asks where and why the method breaks.

Good experimental practice includes:

- one-factor ablations and selected interactions;
- equal tuning budgets for the full model and ablations;
- paired evaluation on identical examples;
- multiple training seeds when optimization is stochastic;
- confidence intervals over the correct unit, such as users rather than rows;
- correction or restraint when many hypotheses are tested;
- learning curves and resource-quality curves;
- slice and qualitative error analysis defined independently of favorable results;
- negative findings and unstable configurations.

For paired predictions, bootstrap the sampling unit and recompute the **difference**, not two unrelated confidence intervals. If rows belonging to one user or document are dependent, resample groups.

<details>
<summary><strong>Python: estimate a paired bootstrap interval for an ablation gain</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(20)
n_groups = 600
rows_per_group = 5
group_id = np.repeat(np.arange(n_groups), rows_per_group)

latent = rng.normal(size=n_groups)
y = np.repeat((latent > 0).astype(int), rows_per_group)
y = np.where(rng.random(len(y)) < 0.12, 1 - y, y)

# Full and ablated model predictions are deliberately correlated because they
# are evaluated on the same rows.
base_signal = np.repeat(latent, rows_per_group)
full_score = 1 / (1 + np.exp(-(base_signal + rng.normal(0, 0.65, len(y)))))
ablated_score = 1 / (1 + np.exp(-(0.78 * base_signal + rng.normal(0, 0.78, len(y)))))


def binary_log_loss(labels, probability):
    probability = np.clip(probability, 1e-9, 1 - 1e-9)
    return -np.mean(
        labels * np.log(probability) + (1 - labels) * np.log(1 - probability)
    )


observed_gain = binary_log_loss(y, ablated_score) - binary_log_loss(y, full_score)
bootstrap_gains = []
for _ in range(2000):
    sampled_groups = rng.integers(0, n_groups, size=n_groups)
    sampled_rows = np.concatenate(
        [np.flatnonzero(group_id == group) for group in sampled_groups]
    )
    gain = (
        binary_log_loss(y[sampled_rows], ablated_score[sampled_rows])
        - binary_log_loss(y[sampled_rows], full_score[sampled_rows])
    )
    bootstrap_gains.append(gain)

lower, upper = np.quantile(bootstrap_gains, [0.025, 0.975])
print("full_model_log_loss_gain", round(observed_gain, 4))
print("group_paired_95_percent_interval", (round(lower, 4), round(upper, 4)))
```

</details>

The interval concerns the sampled group population under this protocol. It does not include uncertainty from changing datasets, preprocessing, hyperparameter search, or training seeds unless those sources are explicitly rerun inside the resampling design.

**Comparison.** A reference rerun verifies an artifact; a clean implementation tests whether the paper specifies the method; a replication tests whether the claim survives a changed setting. Ablations attribute gains, sensitivity maps assumptions, and failure analysis defines the boundary of validity.


### **Communicating Machine Learning Results**

Communication is part of the evidence pipeline. A result that cannot be connected to a population, protocol, comparator, uncertainty estimate, and operational consequence is not decision-ready. The purpose of a technical report is not to display every experiment; it is to make the reasoning auditable.

#### **Technical Reports, Visual Evidence, and Limitations**

A strong report can be organized in the order a reviewer needs:

1. **Decision and recommendation:** what choice is being considered and what the evidence supports.
2. **Scope:** intended population, time period, use, exclusions, and non-goals.
3. **Data and labels:** origin, unit, split, maturity, leakage controls, and limitations.
4. **Protocol:** baselines, candidates, search budget, seeds, selection, metrics, and compute.
5. **Results:** effect sizes, uncertainty, slices, resource metrics, and failure cases.
6. **Robustness of the conclusion:** ablations, sensitivity, alternate specifications, and negative findings.
7. **Operational plan:** packaging, rollout, monitoring, fallback, owners, and retirement criteria.
8. **Limitations and open questions:** what the study does not establish.
9. **Reproduction record:** code, data, environment, commands, and artifact identifiers.

Report **absolute values** and **differences**. A relative improvement can exaggerate a small change:

$$
\text{relative reduction}
=
\frac{e_{\text{baseline}}-e_{\text{candidate}}}
{e_{\text{baseline}}},
\qquad
\text{absolute reduction}
=
e_{\text{baseline}}-e_{\text{candidate}}.
$$

Reducing error from $2.0\%$ to $1.5\%$ is a 25% relative reduction and a 0.5 percentage-point absolute reduction. Both are correct; neither is sufficient without the number of affected decisions, error costs, uncertainty, and slice behavior.

Tables should expose the comparison contract:

| Model | Data and budget | Primary metric | 95% interval | Worst critical slice | p99 latency | Cost | Decision |
|---|---|---:|---:|---:|---:|---:|---|
| Incumbent | version A, budget B | value | interval | value | value | value | retain/promote |
| Candidate | same or justified difference | value | interval | value | value | value | retain/promote |

Visuals should answer a named question:

- learning curves show whether gain comes from data scale;
- calibration plots show probability reliability;
- resource-quality frontiers show feasibility;
- paired difference plots show where one method wins;
- slice matrices show concentrated failure;
- timelines connect deployments, data changes, and incidents;
- failure galleries reveal recurring qualitative patterns.

Avoid decorative plots, truncated axes that distort effects, unlabelled smoothing, and a single best seed. Every figure should state population, aggregation, uncertainty, and direction of improvement. Raw predictions and plotting code should remain in the artifact package.

Limitations are strongest when they identify the exact inference that is unsupported:

- “The evaluation covers existing users in Australia during 2025; it does not establish performance for new users or other countries.”
- “Labels are observed only for reviewed cases, so full-population recall is not identified.”
- “The canary lasted seven days and does not cover annual seasonality.”
- “The study compares equal wall-clock search budgets but not equal energy use.”
- “The confidence interval covers test examples but not retraining-seed variation.”

These statements guide future work. Generic phrases such as “more data may help” do not.

<details>
<summary><strong>Python: build a decision table that refuses incomplete evidence</strong></summary>

```python
import pandas as pd

rows = [
    {
        "model": "incumbent_v7",
        "dataset": "events@2026-04-01",
        "search_budget_gpu_h": 0,
        "primary_metric": 0.781,
        "ci_low": 0.768,
        "ci_high": 0.793,
        "worst_slice_recall": 0.744,
        "p99_latency_ms": 24,
        "cost_per_1000": 0.35,
        "decision": "retain",
    },
    {
        "model": "candidate_v8",
        "dataset": "events@2026-04-01",
        "search_budget_gpu_h": 18,
        "primary_metric": 0.806,
        "ci_low": 0.794,
        "ci_high": 0.818,
        "worst_slice_recall": 0.771,
        "p99_latency_ms": 39,
        "cost_per_1000": 0.48,
        "decision": "canary",
    },
]

required = {
    "model", "dataset", "search_budget_gpu_h", "primary_metric",
    "ci_low", "ci_high", "worst_slice_recall",
    "p99_latency_ms", "cost_per_1000", "decision",
}
table = pd.DataFrame(rows)
missing = required - set(table.columns)
if missing or table[list(required)].isna().any().any():
    raise ValueError(f"Decision evidence is incomplete: {sorted(missing)}")

table["interval"] = table.apply(
    lambda row: f"[{row.ci_low:.3f}, {row.ci_high:.3f}]", axis=1
)
table["feasible"] = (
    (table["worst_slice_recall"] >= 0.75)
    & (table["p99_latency_ms"] <= 50)
    & (table["cost_per_1000"] <= 0.50)
)

print(
    table[
        [
            "model", "dataset", "primary_metric", "interval",
            "worst_slice_recall", "p99_latency_ms",
            "cost_per_1000", "feasible", "decision",
        ]
    ].to_string(index=False)
)
```

</details>

The table exposes an important disagreement: the incumbent fails the newly stated slice requirement, yet its written decision says “retain.” That contradiction should trigger review rather than be silently formatted away. Reporting tools can enforce completeness and arithmetic; accountable people must resolve the decision.

**Comparison.** An experiment log preserves everything that happened; a technical report selects the evidence needed for a claim; an operational runbook specifies what to do; a model or system card summarizes intended use and limitations. These artifacts overlap but serve different readers and decisions.


### **Capstone Research Workflow**

The capstone workflow combines scientific and engineering discipline. It is deliberately expressed as **gates** rather than a linear checklist: a failed gate changes the next action. The team may revise the label, collect data, simplify the model, rerun an experiment, redesign serving, or stop the project.

<div class="diagram-scroll">

![The capstone workflow uses contract, evidence, artifact, release, and operation gates.](assets/capstone-release-gates.svg){fig-alt="Five review gates cover the decision contract, experimental evidence, reproducible artifact, safe release, and continued operation."}

</div>

#### **Gate 1: Decision Contract**

Write one page before modeling:

- decision owner and affected users;
- unit, prediction time, target, horizon, and label maturity;
- action policy, capacity, and fallback;
- current baseline and expected value pathway;
- primary metric, guardrails, critical slices, and hard constraints;
- reasons a deterministic or non-ML method is insufficient;
- stop condition if data or actionability is inadequate.

**Pass evidence:** stakeholders agree that the target and metrics correspond to a real decision and that the project has a maintainable owner.

#### **Gate 2: Data and Experimental Evidence**

Create a versioned dataset with contracts, lineage, point-in-time correctness, group-aware splits, and label-coverage analysis. Implement dummy, heuristic, and classical baselines. Register the candidate protocol before the final comparison when practical:

- candidate families and search budget;
- validation and test policy;
- metrics, thresholds, slices, and uncertainty;
- training seeds and stopping rule;
- resource measurements;
- ablations, sensitivity, and failure analysis.

**Pass evidence:** the candidate improves a credible baseline by a practically meaningful amount, survives uncertainty and critical slices, and does not violate offline constraints.

#### **Gate 3: Reproducible Artifact**

Produce a clean-run package:

- immutable data identifiers and split assignments;
- code commit and environment lock or container digest;
- configuration and seeds;
- one command for training and one for evaluation;
- raw predictions, metrics, plots, and resource logs;
- fitted preprocessing, model signature, and versioned model;
- decision report, limitations, and license information.

Ask another person to reproduce a key table from a clean environment. **Pass evidence:** the result can be traced and materially reproduced without undocumented intervention.

#### **Gate 4: Release**

Test the complete package against serving contracts and replay data. Choose batch, online, or edge deployment from the decision deadline. Define:

- shadow and canary sequence;
- minimum sample and promotion criteria;
- system, data, model, slice, and decision guardrails;
- telemetry fields and model-version logging;
- fallback, rollback, and owner;
- security and access boundaries;
- alert and incident playbooks.

**Pass evidence:** the candidate meets SLOs and guardrails under representative traffic, and the team has demonstrated rollback.

#### **Gate 5: Operate, Learn, and Retire**

Monitor input contracts, freshness, drift, prediction behavior, delayed-label performance, decision outcomes, workload, and resource cost. Review:

- incident history and error-budget consumption;
- feedback and selective-label mechanisms;
- retraining trigger, validation, and promotion history;
- whether the original decision remains relevant;
- whether a simpler rule or newer process now dominates the model;
- retention, archive, and retirement obligations.

**Pass evidence:** continued operation still creates net value under the current contract. A model that no longer has an owner, observable outcome, or safe fallback should be retired even if it once performed well.

<details>
<summary><strong>Python: turn the capstone into explicit release-gate decisions</strong></summary>

```python
from dataclasses import dataclass


@dataclass(frozen=True)
class Evidence:
    decision_contract_signed: bool
    label_coverage: float
    point_in_time_audit_passed: bool
    baseline_gain: float
    lower_confidence_bound_gain: float
    worst_slice_recall: float
    reproducible_clean_run: bool
    p99_latency_ms: float
    availability: float
    rollback_tested: bool
    monitoring_owner_assigned: bool


def review(evidence: Evidence) -> dict:
    gates = {
        "contract": evidence.decision_contract_signed,
        "data": (
            evidence.label_coverage >= 0.95
            and evidence.point_in_time_audit_passed
        ),
        "experimental_evidence": (
            evidence.baseline_gain >= 0.02
            and evidence.lower_confidence_bound_gain > 0
            and evidence.worst_slice_recall >= 0.75
        ),
        "artifact": evidence.reproducible_clean_run,
        "release": (
            evidence.p99_latency_ms <= 50
            and evidence.availability >= 0.999
            and evidence.rollback_tested
            and evidence.monitoring_owner_assigned
        ),
    }
    first_failure = next((name for name, passed in gates.items() if not passed), None)
    return {
        "gates": gates,
        "decision": "APPROVE_CANARY" if first_failure is None else "STOP_AND_REVISE",
        "first_failed_gate": first_failure,
    }


candidate = Evidence(
    decision_contract_signed=True,
    label_coverage=0.982,
    point_in_time_audit_passed=True,
    baseline_gain=0.031,
    lower_confidence_bound_gain=0.012,
    worst_slice_recall=0.77,
    reproducible_clean_run=True,
    p99_latency_ms=43,
    availability=0.9995,
    rollback_tested=False,
    monitoring_owner_assigned=True,
)

print(review(candidate))
```

</details>

The candidate stops at the release gate because rollback has not been demonstrated. The response is not to change `False` to `True`; it is to run the rollback exercise, record its evidence, and review the gate again.

### **Systematic Summary**

| Question | Weak answer | Strong answer |
|---|---|---|
| What are we building? | “A churn model” | A defined decision contract with unit, horizon, action, capacity, users, and constraints |
| Is it better? | Higher score than one untuned model | Controlled gain over credible baselines with uncertainty, slices, and resource evidence |
| Can it be reproduced? | Notebook and model file | Versioned data, code, environment, configuration, predictions, and clean-run commands |
| Can it be deployed? | Endpoint returns a score | Package, schema, load test, safe rollout, SLOs, telemetry, fallback, and rollback |
| Is it still working? | Dashboard looks normal | Owned alerts across system, data, model, delayed outcomes, and decision value |
| Should it retrain? | A feature drifted | Persistent evidence triggers a candidate; the incumbent remains until release gates pass |
| Is the research credible? | Headline number matches | Claim, protocol, budget, uncertainty, ablations, failures, and executable artifacts align |

Research and production emphasize different endpoints but share the same discipline. Research asks whether a claim survives controlled challenge and independent scrutiny. Production asks whether a decision system remains valuable and reliable under changing operation. Both require precise scope, strong baselines, traceable artifacts, uncertainty, failure analysis, and honesty about what the evidence does not establish.

This final chapter closes the machine learning series at the point where an algorithm becomes accountable work: a claim that can be checked, an artifact that can be reproduced, a release that can be reversed, and a system that can be observed, improved, or retired.
